# 06 · Batch Inference with the Approved Tag QA Gate

The latest **Approved** model package is pulled from the Model Registry and run as a **SageMaker Batch Transform**
job over the incoming drafts (games released on/after 2025-01-01, never seen in training).

To measure false approvals, 20% of the drafts are deliberately paired with **another game's tags**, the way copy
written for the wrong game would look. A correct gate rejects them.

In [ ]:
# Pin the SageMaker SDK v2 API used by this project (no-op if already installed)
%pip install -q "sagemaker>=2.232,<3"

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
def find_repo_root():
    """The notebooks import from src/, so the whole repo must be cloned, not just notebooks/."""
    candidates = [os.path.expanduser("~/steam-store-copilot")]
    try:  # the kernel's working directory may have been deleted (e.g. after re-cloning)
        candidates = [os.getcwd(), os.path.dirname(os.getcwd())] + candidates
    except FileNotFoundError:
        os.chdir(os.path.expanduser("~"))
    for path in candidates:
        if os.path.isdir(os.path.join(path, "src")) and os.path.exists(os.path.join(path, "src", "config.py")):
            return path
    raise RuntimeError("Could not find the repo's src/ folder. Clone the full repository in Studio "
                       "(Git > Clone Repository) and open the notebooks from steam-store-copilot/notebooks/.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

import boto3
import sagemaker

from src.config import S3Layout

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()  # LabRole in AWS Academy Learner Lab
layout = S3Layout(bucket=sess.default_bucket())
sm = boto3.client("sagemaker", region_name=region)
print("Region:", region)
print("Role:  ", role)
print("Bucket:", layout.bucket)

In [ ]:
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
from sagemaker import ModelPackage

from src.config import PIPELINE_NAME
from src.data.s3io import read_csv, read_text, write_csv, write_text
from src.monitoring.data_quality import monitor_frame
from src.monitoring.model_quality import batch_quality, read_jsonlines
from src.pipelines.batch import latest_approved_package, make_draft_batch, to_jsonlines

execs = sm.list_pipeline_executions(PipelineName=PIPELINE_NAME, SortBy="CreationTime",
                                    SortOrder="Descending")["PipelineExecutionSummaries"]
last_ok = next(e for e in execs if e["PipelineExecutionStatus"] == "Succeeded")
execution_id = last_ok["PipelineExecutionArn"].split("/")[-1]
incoming = read_csv(f"{layout.pipeline_artifacts}/{execution_id}/data/incoming/incoming.csv", keep_default_na=False)
drafts = make_draft_batch(incoming, mismatch_share=0.2)
print(f"{len(drafts):,} drafts ({drafts['is_mismatch'].sum():,} mismatched) from pipeline execution {execution_id}")

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
batch_prefix = f"{layout.batch}/{run_id}"
write_text(f"{batch_prefix}/input/drafts.jsonl", to_jsonlines(drafts))
write_csv(drafts[["app_id", "is_mismatch"]], f"{batch_prefix}/ground_truth.csv", index=False)
write_csv(monitor_frame(drafts), f"{batch_prefix}/monitor/features.csv", index=False)

package_arn = latest_approved_package(sm)
print("Using model package:", package_arn)
model = ModelPackage(role=role, model_package_arn=package_arn, sagemaker_session=sess)
transformer = model.transformer(instance_count=1, instance_type="ml.m5.xlarge", strategy="MultiRecord",
                                assemble_with="Line", accept="application/jsonlines",
                                output_path=f"{batch_prefix}/output", max_payload=5)
transformer.transform(f"{batch_prefix}/input/drafts.jsonl", content_type="application/jsonlines",
                      split_type="Line", wait=True, logs=False)
print("Transform job:", transformer.latest_transform_job.name)

In [ ]:
out = read_jsonlines(read_text(f"{batch_prefix}/output/drafts.jsonl.out"))
out = out.merge(drafts[["app_id", "is_mismatch"]], on="app_id", how="left")
pd.set_option("display.max_colwidth", 70)
out[["app_id", "true_tags", "predicted_tags", "agreement", "decision", "is_mismatch"]].head(10)

In [ ]:
quality = batch_quality(out, out["is_mismatch"])
write_text(f"{batch_prefix}/quality.json", json.dumps(quality, indent=2))
print(json.dumps(quality, indent=2))

table = pd.crosstab(out["is_mismatch"].map({0: "genuine draft", 1: "mismatched draft"}), out["decision"],
                    normalize="index")
ax = table.plot.barh(stacked=True, color={"approve": "#2e8b57", "reject": "#c2453a", "human_review": "#bbb"},
                     figsize=(10, 3))
ax.set_title("Gate decisions on the incoming batch")
ax.set_xlabel("share of drafts")
plt.show()

In [ ]:
print("Rejected mismatches (correctly caught):")
caught = out[(out["is_mismatch"] == 1) & (out["decision"] == "reject")]
caught[["app_id", "true_tags", "predicted_tags", "agreement"]].head(5)

In [ ]:
write_text(f"{layout.batch}/LATEST", batch_prefix)
print("Batch outputs for monitoring:", batch_prefix)